# Configuring iMD-XR visual selections

This notebook demonstrates the full selection of visualisation options supported by the [NanoVer iMD-XR client](https://irl2.github.io/nanover-docs/installation.html#installing-the-imd-xr-client) that can be controlled from Python and Jupyter via the server shared state.

## Simulation and server setup

We start with the usual server setup, using the provided *Neuraminidase* example, as it has a good mix of features including a polypeptide chain and multiple molecules (protein, ligand, and a couple of ions):

In [1]:
from nanover.app import OmniRunner
from nanover.openmm import OpenMMSimulation

omm_sim = OpenMMSimulation.from_xml_path("../systems/openmm/neuraminidase.openmm.zip")

imd_runner = OmniRunner.with_basic_server(omm_sim, name="visualisations tutorial server")
imd_runner.load(0)
imd_runner.print_basic_info()

Serving "visualisations tutorial server" (ws://localhost:38801), discoverable on all interfaces on port 54545
Available simulations:
[0]: "neuraminidase.openmm"
Switched to [0]: "neuraminidase.openmm"


## Viewing in virtual reality

Connecting the [NanoVer iMD-XR client](https://irl2.github.io/nanover-docs/installation.html#installing-the-imd-xr-client) to the server will present you with a simulation of Neuraminidase, drawn using the standard ball and stick approach. The coloring by default is **CPK**, which colors hydrogens, carbons, oxygens and nitrogens white, grey, red and blue respectively.

![Ball and Stick](images/ball-and-stick.png)

## Modifying the visual selections

The visual selections live in the shared value store of the server, and can be modified either directly on the server or via a remotely connected client.

In other example notebooks we modify the selections using a set of utilities that work directly with the server, but these utilities are also available for remote clients:

In [2]:
from nanover.websocket import NanoverImdClient

client = NanoverImdClient.from_runner(imd_runner)
client.wait_until_minimum_usable_frame()
client.current_frame

<FrameData of bond.pairs[6073, 2], chain.count, chain.names[8], energy.kinetic, energy.potential, frame.index, particle.count, particle.elements[5988], particle.names[5988], particle.positions[5988, 3], particle.residues[5988], residue.chains[395], residue.count, residue.ids[395], residue.names[395], server.timestamp, system.box.vectors[3, 3], system.simulation.counter, system.simulation.name, system.simulation.time>

In [3]:
from nanover.jupyter import NanoverJupyterUtilities

# the server-based utilities from previous examples
utilities = NanoverJupyterUtilities.from_runner(imd_runner)

# the client-based utilities
utilities = NanoverJupyterUtilities.from_client(client)

## The root selection

The **root selection** is a default selection that includes every atom in the simulation. The simplest changes we can make are to specify a named renderer for this root selection, which has the effect of changing how the entire system is rendered in the client:

In [4]:
utilities.selections.update_selection("root", renderer="liquorice")

## Particle selections

We'll use MDAnalysis Universes to make selections easier:

In [ ]:
from nanover.mdanalysis import frame_data_to_mdanalysis

universe = frame_data_to_mdanalysis(utilities.current_frame)
universe

In [ ]:
universe.select_atoms("resname GLU").indices

Here we select a particular residue (by providing a list of atom indices) and specify that it should be rendered as ball and stick (a built-in renderer name in the client):

In [7]:
utilities.selections.update_selection("glu", particle_ids=universe.select_atoms("resname GLU").indices, renderer="ball and stick")

We can make modifications to that existing selection after, for example to hide it entirely:

In [ ]:
utilities.selections.modify_selection("glu", hide=True)

We can also specify what interaction method should be used with interacting with this selection. This can be one of:
* `single`: Standard interaction mode, interacting will grab the nearest atom
* `group`: Grouped interaction mode, interacting with grab the entire selection
* `none`: Disable interactions. If an atom is a member of this selection, it will not be interactable.

## Presets renderers

The simplest way we can change our visualisation is to choose one of the preset visualisers provided in NanoVer. These are fully set up with parameters and colors to allow you to quickly choose an option. Anything possible with presets is also possible with the more detailed setup described below.

The supported presents are:
<table>
    <tr>
        <td style='text-align: center'>
            <code>"ball and stick"</code>
            <img src='./images/presets/ball-and-stick.png'>
        </td>
        <td style='text-align: center'>
            <code>"liquorice"</code>
            <img src='./images/presets/liquorice.png'>
        </td>
        <td style='text-align: center'>
            <code>"noodles"</code>
            <img src='./images/presets/noodles.png'>
        </td>
        <td style='text-align: center'>
            <code>"goodsell"</code>
            <img src='./images/presets/goodsell.png'>
        </td>
    </tr>
    <tr>
        <td style='text-align: center'>
            <code>"cycles"</code>
            <img src='./images/presets/cycles.png'>
        </td>
        <td style='text-align: center'>
            <code>"spline"</code>
            <img src='./images/presets/spline.png'>
        </td>
        <td style='text-align: center'>
            <code>"geometric spline"</code>
            <img src='./images/presets/geometric-spline.png'>
        </td>
        <td style='text-align: center'>
            <code>"cartoon"</code>
            <img src='./images/presets/cartoon.png'>
        </td>
    </tr>
    <tr>
        <td style='text-align: center'>
            <code>"hyperballs"</code>
            <img src='./images/presets/hyperballs.png'>
        </td>
    </tr>
</table>

We choose a visualiser in the same way we alter any other properties of a selection:

In [8]:
utilities.selections.modify_selection("glu", renderer="hyperballs")

## Constructing a renderer

For more control over how the system is displayed, we can construct a renderer. This involves setting the renderer to a dictionary with specific keys, which are instructions on how the client should build a renderer.

A visualiser consists of several parts linked together, which gives a modular construction that allows more control how different values are calculated:

* A **sequence** component (Optional), which calculates sequences of particles indices for drawing as splines.
* A **color** component, which provides a per particle color scheme
* A **scale** component, which provides a per particle scale
* A **width** component, which provides a per particle width (used for visualisers such as ribbons)
* A **render** component, who takes information from the other components and draws something to the screen

All of these parts have a default value, so technically an empty dictionary is a valid renderer:

In [9]:
utilities.selections.update_selection("root", renderer={})

You will see the following visualisation appear:

![](images/examples/empty-dict.png)

This is because the defaults are as follows
* If you don't provide a color, white is used
* If you don't provide a scale, it is 1
* If you don't provide a render, it defaults to ball and stick

Let's try something a little more interesting. Let's pick a color such as cornflower blue, and try it out:

In [10]:
utilities.selections.update_selection(
    "root",
    renderer={
        "color": "CornflowerBlue",
        "render": "liquorice",
    },
)

![](images/examples/cornflower-blue.png)

<div class="alert alert-info">

**Colors:** Colors can be specified in several different ways:
    
   * A string containing a hex color, with or without a preceding hash, such as `'F15H3C'` or `'#FF4E00'`. Googling 'color picker' will give you a widget that allows you to choose a color.
   * Any of the colors defined in the CSS3 standard, such as `'Red'` or `'AliceBlue'`. These will be familiar with those involved in web development, with a full list available [here](https://www.w3schools.com/cssref/css_colors.asp)
   * An array of three floats, defining the R, G and B values from 0 to 1, such as `'[0.3, 0.8, 1.0]'`

Any variable which accepts a color can be set using any of these methods

</div>

For certain arguments such as `color`, `render` and `scale`, providing the name of a specific component (in the above case, we are using the `liquorice` render component) will use that component in the visualiser. A **component** is a part of the visualiser that takes in some data, and outputs some new data. For example, the `cpk` color component takes in the element of each particle, and outputs a color based upon a defined scheme. 

Each component may also has optional arguments which can be modified. Here, instead of just giving the string name of the component, we must provide a dictionary which has a 'type': 'component_name' entry. For example, the `cpk` color component has a parameter `scheme`:

In [12]:
# Use the CPK component with default arguments
utilities.selections.update_selection(
    "root",
    renderer={
        "color": "cpk",
    },
)

![](images/examples/cpk.png)

In [21]:
# Use the CPK component with the 'scheme' argument, which specifies the built-in NanoVer Scheme
utilities.selections.update_selection(
    "root",
    renderer={
        "color": {
            "type": "cpk",
            "scheme": "nanover",
        },
    },
)

![](images/examples/cpk-nanover.png)

In [13]:
# Use the CPK component with the "scheme" argument, but providing your own scheme
utilities.selections.update_selection(
    "root",
    renderer={
        "color": {
            "type": "cpk",
            "scheme": {
                "H": "HoneyDew",
                "C": "DarkSlateGrey",
                "O": "FireBrick",
                "N": "DarkTurquoise",
                "S": "Khaki",
            },
        },
    },
)

![](images/examples/cpk-custom.png)

Once we have a scheme we quite like, we could also play around at modifying the scales of the ball and sticks:

In [14]:
# Use the CPK component with the 'scheme' argument, but providing your own scheme and customising the render
utilities.selections.update_selection(
    "root",
    renderer={
        "color": {
            "type": "cpk",
            "scheme": {
                "H": "HoneyDew",
                "C": "DarkSlateGrey",
                "O": "FireBrick",
                "N": "DarkTurquoise",
                "S": "Khaki",
            },
        },
        "render": {
            "particle.scale": 0.05,
            "bond.scale": 0.02,
            "type": "ball and stick",
        },
    },
)

![](images/examples/cpk-custom2.png)

The Appendix lists all current possible visualisations and their arguments that can be used in NanoVer.

## Examples

In preparation for this section, let's clear all the selections and set our base visualiser to something minimal:

In [16]:
utilities.selections.clear_all()
utilities.selections.update_selection(
    "root",
    renderer={
        "color": {
            "type": "cpk",
            "scheme": "nanover",
        },
        "scale": 0.04,
        "render": "liquorice",
    },
)

![](images/examples/clean.png)

For our example here, we will be creating a selection consisting of ARG-37 and GLU-38:

In [36]:
# Draw the selection in a chunky gold liquorice style
utilities.selections.update_selection(
    "example",
    particle_ids=universe.select_atoms("resid 37 or resid 38").indices,
    renderer={
        "color": "Gold",
        "scale": 0.1,
        "render": "liquorice",
    },
)

![](images/examples/selection1.png)

We can now see our selection! All the atoms that we specified in this selection are now being draw using the renderer specified for our new selection instead.

We can change our selection to be LEU-143, ARG-144 and THR-145:

In [28]:
# Expand the above selection to include LEU-143, ARG-144, and THR-145
utilities.selections.modify_selection("example", particle_ids=universe.select_atoms("resid 37 or resid 38 or resid 143:145").indices)

![](images/examples/selection2.png)

# Appendix - Possible Options for Visualisers

## Parameter Types

Parameters can be of the following types, and can often be specified in different ways:

#### Float

Can be specified as either an integer (`0`, `32`, `-11`) or a decimal (`2.31`, `-0.2`).

#### String

Self explanatory, use a python string

#### Color

Explained in more detail under the Visualisation section, but can be

* A string containing a hex code, with or without preceding #
* A string specifying a CSS3 color name
* A list of three floats, specifying the R, G and B values from 0 to 1

#### Gradient

A gradient is a list of two to seven colors, each specified as above. You can mix and match which styles you use

```python
[
    '#ffff00',
    'SlateGrey',
    [0.1, 0.5, 0.3]
]
```

#### Element-Color Mapping

Defines a map between atomic elements and colors. It can be

* A predefined map. The current options are `jmol` or `nanover`.
* A custom dictionary. The keys of this should be atomic elements, and colors can be specified as above
```python
{
    'H': 'AliceBlue',
    'O': '#ff2e09'
}
```

---

## Color Components

The following components can be used by adding `'color': component_name` to your dictionary. For parameters, you should using

```python
'color': {
    'type': component_name,
    # parameters...
}
```

If you want a solid color, you can simply specify `'color': color` using the color specification defined above

## `cpk`

Color by atomic element

**Parameters**

* `scheme` (Element-Color Mapping) - The mapping to use to transform element to color. Defaults to `jmol`.

## `goodsell`

Goodsell style color scheme. Cycles through a set of 11 pastel colors, coloring each entity uniquely. Colors non carbons in a slightly darker color

## `particle index`

Colors particles by their index in the simulation, with the gradient going from the first particle to the last particle.

**Parameters**

* `gradient` (Gradient) - The gradient to use to color the particles Defaults to a red to blue gradient.

## `residue index`

Colors particles by the index of their residue in the simulation, with the gradient going from the first residue to the last.

**Parameters**

* `gradient` (Gradient) - The gradient to use to color the particles Defaults to a red to blue gradient.

## `residue index in entity`

Colors particles by the index of their residue in their entity, with the gradient going from the first residue of each entity to the last.

**Parameters**

* `gradient` (Gradient) - The gradient to use to color the particles Defaults to a red to blue gradient.

## `residue name`

Colors particles by the name of their residue. Currently hardcoded to use the RCSB colors, but will be modifiable in a future update.

## `secondary structure`

Colors particles by their secondary structure, calculated using the DSSP algorithm. Currently the color is hard coded to:

* Yellow for beta sheets
* Red for pi helices
* Magenta for alpha helices
* Blue for 3-10 helices
* White for everything else

These colors will be modifiable in a future update.

---

## Scale Components

The following components can be used by adding `'scale': component_name` to your dictionary. For parameters, you should using

```python
'scale': {
    'type': component_name,
    # parameters...
}
```

If you want a constant scale, you can simply specify `'scale': scale` using the float specification defined above

## `vdw`

Use the van der Waals radii for each element.

---

## Width Components

The following components can be used by adding `'width': component_name` to your dictionary. For parameters, you should using

```python
'width': {
    'type': component_name,
    # parameters...
}
```

If you want a constant width, you can simply specify `'width': width` using the float specification defined above

## `secondary structure`

Use the following widths based on the calculated secondary structure

* 1 for helices and sheets
* 0 for everything else

This gives width to sheets and helices to allow a ribbon style render

---

## Sequence Components

The following components can be used by adding `'sequence': component_name` to your dictionary. For parameters, you should using

```python
'sequence': {
    'type': component_name,
    # parameters...
}
```

You need to provide a sequence component when dealing with render components such as splines, which you need to tell how to decide which curves to draw

## `entities`

Create sequences that go from the first particle in each entity to the last.

## `polypeptide`

Create sequences of the alpha carbons of subsequent amino acids. All non amino acids are ignored.

---

## Render Components

Render components are responsible for drawing objects to the screen.

The following components can be used by adding `'render': component_name` to your dictionary. For parameters, you should using

```python
'render': {
    'type': component_name,
    # parameters...
}
```

## `ball and stick`

Classic ball and stick render: spheres connected by cylinders.

**Parameters**

* `color` (Color) - Color which is multiplied to whatever is provided from before. Default is white.
* `particle.scale` (Float) - The scaling factor of the balls. Default is 0.1x
* `bond.scale` (Float) - The scaling factor of the bonds. Default is 0.05x

## `liquorice`

Liquorice render, equivalant to a ball and stick render where the radii of the balls and sticks are equal.

**Parameters**

* `color` (Color) - Color which is multiplied to whatever is provided from before. Default is white.
* `scale` (Float) - Scaling factor. Default is 0.05x

## `cycles`

Similar to a liquorice render, but with any rings with between three and six members shaded using a convex hull.

**Parameters**

* `color` (Color) - Color which is multiplied to whatever is provided from before. Default is white.
* `scale` (Float) - Scaling factor. Default is 0.1x

## `goodsell`

Renders standard spheres, but without shading. This also applies an outline based upon the particle's residue, helping to highlight the chains in the molecule.

**Parameters**

* `color` (Color) - Color which is multiplied to whatever is provided from before. Default is white.
* `scale` (Float) - Scaling factor. Default is 0.6x

## `noodles`

Similar to the liquorice render, but curving bonds to give a more organic shape.

**Parameters**

* `color` (Color) - Color which is multiplied to whatever is provided from before. Default is white.
* `scale` (Float) - Scaling factor. Default is 0.1x

## `spline`

Draws a cylinder along each sequence of particles.

**Parameters**

* `scale` (Float) - Scaling factor. Default is 0.1x

## `geometric spline`

Uses a method similar to the cycles renderer to give a chunky geometric representation of a ribbon along each sequence of particles.

**Parameters**

* `scale` (Float) - Scaling factor, describing how chunky the spline is. Default is 0.1x
* `width` (Float) - Scaling factor, describing the width of the spline. Default is 0.1

## `elliptic spline`

Draw a cylindrical spline which can be stretched into an elliptic spline, with a fix to ensure that glitching does not occur when the spline rotates.

**Parameters**

* `scale` (Float) - Scaling factor, describing how chunky the spline is. Default is 0.1x
* `width` (Float) - Scaling factor, describing the width of the spline. Default is 0.1

## `hyperballs`

An implementation of [HyperBalls](https://www.baaden.ibpc.fr/hballs/page9/page11/index.html), in which atoms are joined by organic bonds that taper with distance akin to liquid under surface tension.

**Parameters**

* `color` (Color) - Color which is multiplied to whatever is provided from before. Default is white.
* `scale` (Float) - Scaling factor. Default is 0.1x
* `tension` (Float) - . Default is 0.5.